In [ ]:
import os
import pandas as pd
import numpy as np

from pyoceanmap import (
    merge_txt_to_csv,
    generate_arctic_grid,
    add_depth_to_grid,
    compute_dynamic_height,
    compute_freshwater,
    objective_map,
)

# pyoceanmap: end-to-end demo (Arctic UDASH case study)

Reproduces the Arctic dynamic-height mapping case study using the installed `pyoceanmap`
package rather than ad hoc inline functions. Requires the real UDASH `.txt` files under
`../data/` and the IBCAO NetCDF under `../data_preparation/` (both too large to ship in this
repository — see the README for download instructions). This notebook is illustrative and is
not run in CI.

In [ ]:
out = merge_txt_to_csv("./data/", "./data.csv")
print("Saved:", out)

In [ ]:
data = pd.read_csv("data.csv")
print(data.head())

In [ ]:
generate_arctic_grid(
    output_file="grid.csv",
    dx=50000   # 50 km spacing
)

In [ ]:
grid = pd.read_csv("grid.csv")
print(grid.head())

In [ ]:
grid_csv = add_depth_to_grid(
    grid_csv="grid.csv",
    output_csv="grid.csv",
    nc_file="./data_preparation/IBCAO_v4_2_13_400m.nc",
)

In [ ]:
grid = pd.read_csv("grid.csv")
print(grid.head())

In [ ]:
compute_dynamic_height(
    input_csv="./data.csv",
    output_csv="data_points.csv"
)

In [ ]:
data_points = pd.read_csv("data_points.csv")
print(data_points.head())

In [ ]:
compute_freshwater(
    input_csv="./data.csv",
    output_csv="freshwater_points.csv"
)

In [ ]:
fw_points = pd.read_csv("freshwater_points.csv")
print(fw_points.head())

In [ ]:
objective_map(
    data_csv="./data_points.csv",
    grid_csv="./grid.csv",
    output_csv="mapped_dh.csv",
    observable="Surf_DH",
    target_time="2013-09-01"
)

Plotting below uses `mpl_toolkits.basemap`, which is only needed for this example (`pip install basemap`) — it is not a `pyoceanmap` package dependency.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap

# -------------------------------
# Setup projection
# -------------------------------
m = Basemap(projection='nplaea', boundinglat=70, lon_0=0,
            resolution='l', round=True)

fig, axes = plt.subplots(1, 2, figsize=(12,6),
                         gridspec_kw={'wspace':0.05})

# ==========================================================
# SUBPLOT 1 — Observations
# ==========================================================
df1 = pd.read_csv("data_points.csv")

ax1 = axes[0]
m.ax = ax1
m.drawcoastlines(linewidth=1.2)
m.drawparallels([80], labels=[1,0,0,0], fontsize=7)
m.drawmeridians([-180,-90,0,90], labels=[0,0,0,1], fontsize=7)

x1, y1 = m(df1["Longitude"].values, df1["Latitude"].values)

sc1 = m.scatter(x1, y1, c=df1["Surf_DH"].values,
                s=20, cmap="YlGnBu",
                vmin=0, vmax=0.8)

ax1.set_title("Observed Surface DH (2009–2015)")

# ==========================================================
# SUBPLOT 2 — Gridded DH
# ==========================================================
df2 = pd.read_csv("mapped_dh.csv")
df2 = df2.dropna(subset=["Surf_DH", "Surf_DH_err"])

ax2 = axes[1]
m.ax = ax2
m.drawcoastlines(linewidth=1.2)
m.drawparallels([80], labels=[0,0,0,0], fontsize=7)
m.drawmeridians([-180,-90,0,90], labels=[0,0,0,1], fontsize=7)

x2, y2 = m(df2["Longitude"].values, df2["Latitude"].values)

sc2 = m.scatter(x2, y2, c=df2["Surf_DH"].values,
                s=20, cmap="YlGnBu",
                vmin=0, vmax=0.8)

ax2.set_title("Gridded Surface DH – Jan 2011")

# ==========================================================
# COMMON COLORBAR
# ==========================================================
cbar = fig.colorbar(sc2, ax=axes.ravel().tolist(),
                    orientation="vertical", shrink=0.8)
cbar.set_label("Dynamic Height (m)")

# plt.savefig("figures/Observed_vs_Gridded_DH_201101.png", dpi=300, bbox_inches="tight")
plt.show()

